# Лаборатори 6: Нэгдүгээр эрэмбийн логик
pyDatalog ба Prolog ашиглан гэр бүлийн мэдлэгийн сан байгуулах

Алхам бүр тусдаа markdown гарчигтай. Алхам бүрийн "Шалгах" cell-ийн үр дүн доор нь үлдсэн.

## Алхам 1: Орчин бэлтгэх

In [1]:
%pip install pyDatalog

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pyDatalog import pyDatalog

# Cell дахин ажиллуулахад мэдлэгийн сан давхардахгүй байхын тулд заавал дуудна
pyDatalog.clear()

# Жижиг жишээ: bird, flies
pyDatalog.create_terms('bird, flies, X')
+bird('tweety')
+bird('sparrow')
flies(X) <= bird(X)
print(flies(X))

X      
-------
sparrow
tweety 


In [3]:
# Шалгах: flies(X) хоёр утга (sparrow, tweety) буцаана.
# Энэ cell-ийг хоёр удаа ажиллуулсан ч хоёр утгатай хэвээр байх ёстой.
pyDatalog.clear()
pyDatalog.create_terms('bird, flies, X')
+bird('tweety')
+bird('sparrow')
flies(X) <= bird(X)
print(sorted(r[0] for r in flies(X).data))

['sparrow', 'tweety']


In [4]:
# Жишээ баримтыг арилгаад, дараагийн алхмуудад хэрэглэх нэрсийг зарлана
pyDatalog.clear()

pyDatalog.create_terms('male, female, parent, person, has_child, childless, '
                       'father, mother, son, daughter, grandparent, sibling, brother, '
                       'sister, uncle, aunt, cousin, ancestor, descendant')
pyDatalog.create_terms('X, Y, Z, P, Q')


def count(q):
    """Асуултын хариултын тоо. pyDatalog.ask нь хариу олдохгүй бол None буцаадаг, тэр үед 0."""
    r = pyDatalog.ask(q)
    return 0 if r is None else len(r.answers)


def answers(q):
    """Нэг хувьсагчтай асуултын хариултыг эрэмбэлж буцаана."""
    r = pyDatalog.ask(q)
    return [] if r is None else sorted(a[0] for a in r.answers)


def check(name, got, expected):
    ok = got == expected
    print(("OK    " if ok else "АЛДАА ") + f"{name}: {got}" + ("" if ok else f"  (хүлээгдсэн: {expected})"))

## Алхам 2: Гэр бүлийн мэдлэгийн сан (баримт)

In [5]:
# male(X): 8 баримт
for name in ['bataa', 'bold', 'tomor', 'dorj', 'zolboo', 'munkh', 'temuulen', 'bat']:
    +male(name)

# female(X): 7 баримт
for name in ['saraa', 'oyuna', 'tsetseg', 'ganaa', 'anar', 'naraa', 'enkhee']:
    +female(name)

# parent(P, C): P нь C-ийн эцэг эх. Нэг гэр бүлд нэг давталт (эцэг эх хос, хүүхдүүд)
families = [
    (['bataa', 'saraa'],  ['bold', 'oyuna', 'tomor']),
    (['bold', 'tsetseg'], ['anar', 'zolboo']),
    (['dorj', 'oyuna'],   ['munkh', 'naraa']),
    (['tomor', 'ganaa'],  ['enkhee']),
    (['bat', 'anar'],     ['temuulen']),
]
for parents, children in families:
    for p in parents:
        for c in children:
            +parent(p, c)

In [6]:
# Шалгах
check("count parent", count("parent(X, Y)"), 18)
check("count male", count("male(X)"), 8)
check("count female", count("female(X)"), 7)
check("parent('bataa', X)", answers("parent('bataa', X)"), ['bold', 'oyuna', 'tomor'])

OK    count parent: 18
OK    count male: 8
OK    count female: 7
OK    parent('bataa', X): ['bold', 'oyuna', 'tomor']


## Алхам 3: Энгийн дүрэм

In [7]:
# father: ∀x ∀y (parent(x, y) ∧ male(x) → father(x, y))
father(X, Y) <= parent(X, Y) & male(X)

# mother: ∀x ∀y (parent(x, y) ∧ female(x) → mother(x, y))
mother(X, Y) <= parent(X, Y) & female(X)

# son: ∀x ∀y (parent(y, x) ∧ male(x) → son(x, y)), X нь Y-ийн хүү
son(X, Y) <= parent(Y, X) & male(X)

# daughter: ∀x ∀y (parent(y, x) ∧ female(x) → daughter(x, y)), X нь Y-ийн охин
daughter(X, Y) <= parent(Y, X) & female(X)

# person: эрэгтэй эсвэл эмэгтэй хэн бүхэн. Нэг предикатад хоёр дүрэм нь "эсвэл"
person(X) <= male(X)
person(X) <= female(X)

person(X) <= female(X)

In [8]:
# Шалгах
check("father(X, 'anar')", answers("father(X, 'anar')"), ['bold'])
check("mother(X, 'bold')", answers("mother(X, 'bold')"), ['saraa'])
check("son(X, 'oyuna')", answers("son(X, 'oyuna')"), ['munkh'])
check("daughter(X, 'oyuna')", answers("daughter(X, 'oyuna')"), ['naraa'])
check("count father", count("father(X, Y)"), 9)
check("count mother", count("mother(X, Y)"), 9)
check("count person", count("person(X)"), 15)

OK    father(X, 'anar'): ['bold']
OK    mother(X, 'bold'): ['saraa']
OK    son(X, 'oyuna'): ['munkh']
OK    daughter(X, 'oyuna'): ['naraa']
OK    count father: 9
OK    count mother: 9
OK    count person: 15


## Алхам 4: Өвөө эмээ ба ах дүү

In [9]:
# grandparent: ∀x ∀y ∀z (parent(x, z) ∧ parent(z, y) → grandparent(x, y))
# Z нь дүрмийн бие дээр л байгаа тул ∃ гэж уншина: "ийм Z оршин байвал"
grandparent(X, Y) <= parent(X, Z) & parent(Z, Y)

# sibling: ∀x ∀y ∀p (parent(p, x) ∧ parent(p, y) ∧ x ≠ y → sibling(x, y))
# X != Y нөхцөлгүй бол хүн бүр өөрийнхөө ах дүү болно
sibling(X, Y) <= parent(P, X) & parent(P, Y) & (X != Y)

sibling(X,Y) <= parent(P,X)&parent(P,Y)&!=(X,Y)

In [10]:
# Шалгах
check("grandparent(X, 'temuulen')", answers("grandparent(X, 'temuulen')"), ['bold', 'tsetseg'])
check("count grandparent", count("grandparent(X, Y)"), 12)
check("sibling(X, 'munkh')", answers("sibling(X, 'munkh')"), ['naraa'])
check("sibling(X, 'bold')", answers("sibling(X, 'bold')"), ['oyuna', 'tomor'])
check("count sibling", count("sibling(X, Y)"), 10)

OK    grandparent(X, 'temuulen'): ['bold', 'tsetseg']
OK    count grandparent: 12
OK    sibling(X, 'munkh'): ['naraa']
OK    sibling(X, 'bold'): ['oyuna', 'tomor']
OK    count sibling: 10


In [11]:
# X != Y нөхцөлийг түр хасаж үзэх (тусдаа нэртэй дүрмээр)
pyDatalog.create_terms('sibling_bad')
sibling_bad(X, Y) <= parent(P, X) & parent(P, Y)
check("count sibling_bad (X != Y-гүй)", count("sibling_bad(X, Y)"), 19)
print("sibling_bad(X, X):", sorted(a for a in pyDatalog.ask("sibling_bad(X, Y)").answers if a[0] == a[1]))

OK    count sibling_bad (X != Y-гүй): 19
sibling_bad(X, X): [('anar', 'anar'), ('bold', 'bold'), ('enkhee', 'enkhee'), ('munkh', 'munkh'), ('naraa', 'naraa'), ('oyuna', 'oyuna'), ('temuulen', 'temuulen'), ('tomor', 'tomor'), ('zolboo', 'zolboo')]


## Алхам 5: Ураг төрлийн дүрмүүд

In [12]:
# brother: X нь Y-ийн эрэгтэй ах дүү
brother(X, Y) <= sibling(X, Y) & male(X)

# sister: X нь Y-ийн эмэгтэй ах дүү
sister(X, Y) <= sibling(X, Y) & female(X)

# uncle: X нь Y-ийн эцэг эхийн эрэгтэй ах дүү (зөвхөн цусан төрлийн авга)
uncle(X, Y) <= parent(P, Y) & brother(X, P)

# aunt: X нь Y-ийн эцэг эхийн эмэгтэй ах дүү
aunt(X, Y) <= parent(P, Y) & sister(X, P)

# cousin: X ба Y-ийн эцэг эхүүд ах дүү
cousin(X, Y) <= parent(P, X) & parent(Q, Y) & sibling(P, Q)

cousin(X,Y) <= parent(P,X)&parent(Q,Y)&sibling(P,Q

In [13]:
# Шалгах
check("brother(X, 'oyuna')", answers("brother(X, 'oyuna')"), ['bold', 'tomor'])
check("sister(X, 'zolboo')", answers("sister(X, 'zolboo')"), ['anar'])
check("uncle(X, 'anar')", answers("uncle(X, 'anar')"), ['tomor'])
check("aunt(X, 'anar')", answers("aunt(X, 'anar')"), ['oyuna'])
check("uncle(X, 'temuulen')", answers("uncle(X, 'temuulen')"), ['zolboo'])
check("cousin(X, 'enkhee')", answers("cousin(X, 'enkhee')"), ['anar', 'munkh', 'naraa', 'zolboo'])
check("cousin(X, 'anar')", answers("cousin(X, 'anar')"), ['enkhee', 'munkh', 'naraa'])
check("count cousin", count("cousin(X, Y)"), 16)

OK    brother(X, 'oyuna'): ['bold', 'tomor']
OK    sister(X, 'zolboo'): ['anar']
OK    uncle(X, 'anar'): ['tomor']
OK    aunt(X, 'anar'): ['oyuna']
OK    uncle(X, 'temuulen'): ['zolboo']
OK    cousin(X, 'enkhee'): ['anar', 'munkh', 'naraa', 'zolboo']
OK    cousin(X, 'anar'): ['enkhee', 'munkh', 'naraa']
OK    count cousin: 16


## Алхам 6: Рекурсив дүрэм

In [14]:
# Суурь дүрэм: шууд эцэг эх бол өвөг
# ∀x ∀y (parent(x, y) → ancestor(x, y))
ancestor(X, Y) <= parent(X, Y)

# Рекурсив дүрэм: X нь Z-ийн эцэг эх, Z нь Y-ийн өвөг бол X нь Y-ийн өвөг
# ∀x ∀y (∃z (parent(x, z) ∧ ancestor(z, y)) → ancestor(x, y))
ancestor(X, Y) <= parent(X, Z) & ancestor(Z, Y)

# descendant: ancestor-ийг эсрэг чиглэлд хэрэглэнэ
descendant(X, Y) <= ancestor(Y, X)

descendant(X,Y) <= ancestor(Y,X)

**Хэлхээ:** `ancestor(bataa, temuulen)` үнэн болохыг батлах дөрвөн хүн:

`bataa → bold → anar → temuulen`

- `parent(bataa, bold)` ба `ancestor(bold, temuulen)` ⇒ `ancestor(bataa, temuulen)`
- `parent(bold, anar)` ба `ancestor(anar, temuulen)` ⇒ `ancestor(bold, temuulen)`
- `parent(anar, temuulen)` ⇒ `ancestor(anar, temuulen)` (суурь дүрэм)

In [15]:
# Шалгах
check("ancestor(X, 'temuulen')", answers("ancestor(X, 'temuulen')"),
      ['anar', 'bat', 'bataa', 'bold', 'saraa', 'tsetseg'])
check("count descendant(X, 'bataa')", count("descendant(X, 'bataa')"), 9)
check("ancestor(X, 'anar')", answers("ancestor(X, 'anar')"), ['bataa', 'bold', 'saraa', 'tsetseg'])
check("count ancestor", count("ancestor(X, Y)"), 32)

OK    ancestor(X, 'temuulen'): ['anar', 'bat', 'bataa', 'bold', 'saraa', 'tsetseg']
OK    count descendant(X, 'bataa'): 9
OK    ancestor(X, 'anar'): ['bataa', 'bold', 'saraa', 'tsetseg']
OK    count ancestor: 32


## Алхам 7: Асуулт ба үгүйсгэл

In [16]:
pyDatalog.create_terms('great_grandparent')

# has_child: X нь ямар нэг хүний эцэг эх. Y нь зөвхөн бие дээр байна (∃y)
has_child(X) <= parent(X, Y)

# childless: X нь person бөгөөд has_child биш.
# Үгүйсгэлийг хувьсагч X-ийг person(X)-ээр холбосны дараа хэрэглэнэ
childless(X) <= person(X) & ~has_child(X)

# great_grandparent: X нь Z-ийн эцэг эх, Z нь Y-ийн өвөө эмээ
great_grandparent(X, Y) <= parent(X, Z) & grandparent(Z, Y)

great_grandparent(X,Y) <= parent(X,Z)&grandparent(

In [17]:
# Шалгах
check("count childless", count("childless(X)"), 5)
check("childless(X)", answers("childless(X)"), ['enkhee', 'munkh', 'naraa', 'temuulen', 'zolboo'])
check("count childless('bataa')", count("childless('bataa')"), 0)
check("count childless('temuulen')", count("childless('temuulen')"), 1)
check("great_grandparent(X, 'temuulen')", answers("great_grandparent(X, 'temuulen')"), ['bataa', 'saraa'])

OK    count childless: 5
OK    childless(X): ['enkhee', 'munkh', 'naraa', 'temuulen', 'zolboo']
OK    count childless('bataa'): 0
OK    count childless('temuulen'): 1
OK    great_grandparent(X, 'temuulen'): ['bataa', 'saraa']


### Алхам 7: асуултын хүснэгт (программын хариу)

In [18]:
r = pyDatalog.ask("cousin(X, Y)")
has_cousin = sorted({a[0] for a in r.answers})
r = pyDatalog.ask("grandparent(X, Y)")
has_grandchild = sorted({a[0] for a in r.answers})

print("1. Болдын хүүхдүүд                  :", answers("parent('bold', X)"))
print("2. Temuulen-ийн өвөө эмээ            :", answers("grandparent(X, 'temuulen')"))
print("3. Munkh-ийн авга ах нар             :", answers("uncle(X, 'munkh')"))
print("4. Enkhee-ийн үеэлүүд                :", answers("cousin(X, 'enkhee')"))
print("5. Үеэлтэй хүмүүс                    :", len(has_cousin), has_cousin)
print("6. Хүүхэдгүй хүмүүс                  :", answers("childless(X)"))
print("7. Temuulen-ийн эцэг эхийн өвөө эмээ :", answers("great_grandparent(X, 'temuulen')"))
print("8. Ач, зээтэй хүмүүс                 :", has_grandchild)

1. Болдын хүүхдүүд                  : ['anar', 'zolboo']
2. Temuulen-ийн өвөө эмээ            : ['bold', 'tsetseg']
3. Munkh-ийн авга ах нар             : ['bold', 'tomor']
4. Enkhee-ийн үеэлүүд                : ['anar', 'munkh', 'naraa', 'zolboo']
5. Үеэлтэй хүмүүс                    : 5 ['anar', 'enkhee', 'munkh', 'naraa', 'zolboo']
6. Хүүхэдгүй хүмүүс                  : ['enkhee', 'munkh', 'naraa', 'temuulen', 'zolboo']
7. Temuulen-ийн эцэг эхийн өвөө эмээ : ['bataa', 'saraa']
8. Ач, зээтэй хүмүүс                 : ['bataa', 'bold', 'saraa', 'tsetseg']


| № | Асуулт | Хариулт |
|---|---|---|
| 1 | Болдын хүүхдүүд хэн бэ? | anar, zolboo |
| 2 | Temuulen-ийн өвөө эмээ хэн бэ? | bold, tsetseg |
| 3 | Munkh-ийн авга ах нар хэн бэ? | bold, tomor |
| 4 | Enkhee-ийн үеэлүүд хэн бэ? | anar, munkh, naraa, zolboo |
| 5 | Үеэлтэй хүмүүс хэд вэ, хэн бэ? | 5 хүн: anar, enkhee, munkh, naraa, zolboo |
| 6 | Хүүхэдгүй хүмүүс хэн бэ? | enkhee, munkh, naraa, temuulen, zolboo |
| 7 | Temuulen-ийн эцэг эхийн өвөө эмээ хэн бэ? | bataa, saraa |
| 8 | Ач, зээтэй хүмүүс хэн бэ? | bataa, bold, saraa, tsetseg |

Хариу бүрийг гэр бүлийн зургаас гараар шалгахад программын хариутай таарсан.

## Нэмэлт даалгавар 1: Нөхөр эхнэр ба хуримын ураг

In [19]:
pyDatalog.create_terms('spouse, uncle_in_law, aunt_in_law')

# spouse: X ба Y нь нэг хүүхдийн эцэг эх бөгөөд өөр хүн
spouse(X, Y) <= parent(X, Z) & parent(Y, Z) & (X != Y)

# uncle_in_law: X нь Y-ийн авга эгчийн нөхөр
uncle_in_law(X, Y) <= spouse(X, P) & aunt(P, Y)

# aunt_in_law: X нь Y-ийн авга ахын эхнэр
aunt_in_law(X, Y) <= spouse(X, P) & uncle(P, Y)

aunt_in_law(X,Y) <= spouse(X,P)&uncle(P,Y)

In [20]:
# Шалгах
check("count spouse", count("spouse(X, Y)"), 10)
check("uncle_in_law", sorted(pyDatalog.ask("uncle_in_law(X, Y)").answers),
      [('dorj', 'anar'), ('dorj', 'enkhee'), ('dorj', 'zolboo')])
check("count aunt_in_law", count("aunt_in_law(X, Y)"), 7)
print("aunt_in_law:", sorted(pyDatalog.ask("aunt_in_law(X, Y)").answers))

OK    count spouse: 10
OK    uncle_in_law: [('dorj', 'anar'), ('dorj', 'enkhee'), ('dorj', 'zolboo')]
OK    count aunt_in_law: 7
aunt_in_law: [('ganaa', 'anar'), ('ganaa', 'munkh'), ('ganaa', 'naraa'), ('ganaa', 'zolboo'), ('tsetseg', 'enkhee'), ('tsetseg', 'munkh'), ('tsetseg', 'naraa')]


## Нэмэлт даалгавар 2: Шинэ гишүүн нэмэх

In [21]:
# Дүрмийг огт өөрчлөхгүйгээр гурван баримт нэмнэ
+male('dulguun')
+parent('bat', 'dulguun')
+parent('anar', 'dulguun')

check("sibling(X, 'temuulen')", answers("sibling(X, 'temuulen')"), ['dulguun'])
check("ancestor(X, 'dulguun')", answers("ancestor(X, 'dulguun')"),
      ['anar', 'bat', 'bataa', 'bold', 'saraa', 'tsetseg'])

OK    sibling(X, 'temuulen'): ['dulguun']
OK    ancestor(X, 'dulguun'): ['anar', 'bat', 'bataa', 'bold', 'saraa', 'tsetseg']


### FOL томьёо

| Дүрэм | FOL |
|---|---|
| `father` | `∀x ∀y (parent(x,y) ∧ male(x) → father(x,y))` |
| `grandparent` | `∀x ∀y (∃z (parent(x,z) ∧ parent(z,y)) → grandparent(x,y))` |
| `ancestor` (суурь) | `∀x ∀y (parent(x,y) → ancestor(x,y))` |
| `ancestor` (рекурсив) | `∀x ∀y (∃z (parent(x,z) ∧ ancestor(z,y)) → ancestor(x,y))` |

**Бие дэх хувьсагч ∃ юм уу ∀ юм уу?** Толгойд байхгүй, зөвхөн бие дээр гарсан хувьсагч (`Z`) нь **∃**: "ийм Z оршин байвал дүгнэлт үнэн". Эквивалент хэлбэрээр ∃-г импликацийн өмнөх талаас гаргавал ∀ болно: `∀x ∀y ∀z (parent(x,z) ∧ parent(z,y) → grandparent(x,y))`. Хоёр хэлбэр ижил утгатай. `ancestor` нь суурь ба рекурсив гэсэн **хоёр дүрэм хамт** байж утга өгнө.

### "Хүн бүр эцэг эхтэй"

`∀y (person(y) → ∃x parent(x, y))`

Энэ мэдлэгийн санд өгүүлбэр **худал**. **bataa** нь `person` боловч `parent(x, bataa)` гэсэн баримт, дүрэм байхгүй. Closed-world assumption-ээр "bataa-ийн эцэг эх байхгүй" гэж үзнэ. Мөн saraa, tsetseg, dorj, ganaa, bat-д эцэг эх байхгүй.

In [22]:
# Нотолгоо: эцэг эхгүй хүмүүс
r = pyDatalog.ask("person(X)")
people = sorted(a[0] for a in r.answers)
no_parent = [p for p in people if count(f"parent(X, '{p}')") == 0]
print("Эцэг эхгүй хүмүүс:", no_parent)

Эцэг эхгүй хүмүүс: ['bat', 'bataa', 'dorj', 'ganaa', 'saraa', 'tsetseg']


## Үгээр хариулах асуултууд

**Алхам 4: `X != Y`-г хасвал 19 болно. Нэмэгдсэн 9 нь юу вэ, яагаад 9 вэ?**
`sibling(X, X)` гэсэн хос, өөрөөр хэлбэл хүн бүр өөрийнхөө ах дүү болно. Эцэг эхтэй 9 хүн (bold, oyuna, tomor, anar, zolboo, munkh, naraa, enkhee, temuulen) бүр нэг ийм хос үүсгэнэ. 10 + 9 = 19. Эцэг эхгүй 6 хүн (bataa, saraa, tsetseg, dorj, ganaa, bat) тоологдохгүй.

**Алхам 7: unsafe negation ба яагаад `~parent(X, Y)`-г шууд хэрэглэхгүй вэ?**
Үгүйсгэл нь хувьсагч холбогдсон үед л утгатай. X холбогдоогүй үед `~has_child(X)` нь хязгааргүй олонлог болно. Тиймээс `person(X)`-ээр X-г эхлээд холбоно. `~parent(X, Y)` нь X ба Y хоёр хувьсагчтай, "X нь Y-ийн эцэг эх биш" гэсэн утгатай. Бидэнд хэрэгтэй нь "X нь **ямар ч** хүний эцэг эх биш". Тиймээс Y-г бие дотор нууж `has_child(X)` гэж тусад нь тодорхойлсон.

**Алхам 8: Prolog-д sibling давхардаж, pyDatalog-д яагаад давхардаагүй вэ?**
Prolog нь хариу бүрийг батлах зам бүрээр нь гаргадаг. Oyuna, bold хоёр хоёр эцэг эхтэй (bataa, saraa) тул `sibling(oyuna, bold)` хоёр замаар батлагдаж 2 удаа гарна. pyDatalog нь Datalog тул хариултыг олонлог болгон буцаадаг, давхардал автоматаар арилна.

**Нэмэлт 4: spouse хүүхэдгүй хосыг яагаад илрүүлж чадахгүй вэ?**
`spouse` нь "нэг хүүхдийн хоёр эцэг эх" гэсэн дүрмээр тодорхойлогддог. Хүүхэдгүй хос нэг ч `parent` баримтыг хуваалцдаггүй. Засахын тулд `married(X, Y)` гэсэн баримтыг мэдлэгийн санд шууд нэмнэ.